In [17]:
text = open('input.txt', 'r').read()
text[:10]

'First Citi'

In [18]:
# TODO try words instead of chars
unique_chars = sorted(list(set(text)))

In [19]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [20]:
tokenizer_map = {char : i for i,char in enumerate(unique_chars)}
untokenizer_map = {i : char for i,char in enumerate(unique_chars)}
encode = lambda string : [tokenizer_map[char] for char in string]
decode = lambda token_list: ''.join(
    untokenizer_map[int(token)] for token in token_list
)

In [21]:
import torch
full_text = torch.tensor(encode(text), dtype=torch.int64, device=device) # must be int64

In [22]:
# TODO tune
CONTEXT_WINDOW_SIZE = 32
N_BATCHES = 16
LEARNING_RATE = 1e-3
LEARNING_RATE_DECAY = 0.999
EPOCHS = int(5e3)
N_ATTENTION_HEADS = 4
N_FEATURE_DIMS = 64
QUERY_SIZE = 16
N_UNIQUE_CHARS = len(unique_chars)
SEED = 42
N_HIDDEN_NEURONS = 200
DROPOUT = 0.01
DEBUG = False
# CONTEXT_WINDOW_SIZE = 3
# N_BATCHES = 3
# LEARNING_RATE = 1e-3
# LEARNING_RATE_DECAY = 0.999
# EPOCHS = int(1e0)
# N_ATTENTION_HEADS = 1
# N_FEATURE_DIMS = 4
# QUERY_SIZE = 4
# N_UNIQUE_CHARS = 65
# SEED = 42
# N_HIDDEN_NEURONS = 200
# DROPOUT = 0.01
# DEBUG = True

In [23]:
dev_cutoff = int(0.9 * len(full_text)) # TODO add test split
train_data = full_text[:dev_cutoff]
dev_data = full_text[dev_cutoff:]

In [24]:
def GetRandomBatch(data):
    batch_start_indices = torch.randint(low=0, high=len(data) - CONTEXT_WINDOW_SIZE, size=(N_BATCHES,), device=device)
    offsets = torch.arange(CONTEXT_WINDOW_SIZE, device=device)
    inputs = data[batch_start_indices[:, None] + offsets]
    labels = data[batch_start_indices[:, None] + offsets + 1]
    if DEBUG: print("BATCH", inputs.shape, labels.shape)
    return inputs, labels


In [25]:
def LayerNorm(x):
    mean = x.mean(dim=-1, keepdim=True)
    std = x.std(dim=-1, keepdim=True)
    return (x - mean) / (std + 1e-5)

In [26]:
class CheckError(RuntimeError):
    pass


def _normalize_for_check(x):
    # Treat torch.Size, tuple, and list shapes as equivalent.
    if isinstance(x, (list, tuple)):
        return tuple(x)

    # Avoid requiring torch import just for this helper.
    if type(x).__name__ == "Size" and type(x).__module__.startswith("torch"):
        return tuple(x)

    return x


def check_eq(left, right, left_name="left", right_name="right"):
    left_cmp = _normalize_for_check(left)
    right_cmp = _normalize_for_check(right)

    if left_cmp != right_cmp:
        raise CheckError(
            f"CHECK_EQ failed: {left_name} == {right_name}\n"
            f"  {left_name}:  {left!r}\n"
            f"  {right_name}: {right!r}"
        )

In [27]:
import math
import torch.nn as nn

class AttentionHead():
    def __init__(self):
        self.query_matrix = torch.randn((N_FEATURE_DIMS, QUERY_SIZE), device=device)
        self.key_matrix = torch.randn(self.query_matrix.shape, device=device)
        self.value_matrix_up = torch.randn((N_FEATURE_DIMS, QUERY_SIZE), device=device)
        self.value_matrix_down = torch.randn((QUERY_SIZE, N_FEATURE_DIMS), device=device)
        self.params = [self.query_matrix, self.key_matrix, self.value_matrix_up, self.value_matrix_down]
        for param in self.params:
            param.requires_grad = True
        # self.attention_matrix = torch.empty((CONTEXT_WINDOW_SIZE, CONTEXT_WINDOW_SIZE))

    def attend(self, feature_vectors):
        check_eq(feature_vectors.shape, [CONTEXT_WINDOW_SIZE, N_FEATURE_DIMS])
        if DEBUG: print("ATTENTION")
        query_vectors = feature_vectors @ self.query_matrix  # TODO pytorch vectorize
        key_vectors = feature_vectors @ self.key_matrix # TODO pytorch vectorize
        attention_matrix = query_vectors @ key_vectors.t()
        if DEBUG: print(query_vectors.shape, key_vectors.shape, attention_matrix.shape)
        
        attention_matrix /= math.sqrt(QUERY_SIZE)
        if DEBUG: print(attention_matrix)
        for row in range(attention_matrix.shape[0]):
            for col in range(attention_matrix.shape[1]):
                if (col > row):
                    attention_matrix[row,col] = -math.inf;
        attention_matrix = torch.softmax(attention_matrix, dim=1)
        enrichment_stack = torch.zeros_like(feature_vectors)
        for row in range(attention_matrix.shape[0]):
            for col in range(attention_matrix.shape[1]):
                enrichment_stack[row] += attention_matrix[row,col] * self.value_matrix_up @ self.value_matrix_down @ feature_vectors[col]
        if DEBUG: print(feature_vectors.shape, enrichment_stack.shape)
        return (feature_vectors + enrichment_stack).view(-1, CONTEXT_WINDOW_SIZE, N_FEATURE_DIMS)

In [28]:
def XavierFactor(param):
    return math.sqrt(2 / (param.shape[0] + param.shape[1]))

class FeedForward():
    def __init__(self):
        torch.manual_seed(SEED) # for reproducibility
        self.W1 = torch.randn((N_ATTENTION_HEADS * N_FEATURE_DIMS, N_HIDDEN_NEURONS), device=device)
        self.W1 *= XavierFactor(self.W1)
        self.b1 = torch.zeros((N_HIDDEN_NEURONS,), device=device)
        self.W2 = torch.randn((N_HIDDEN_NEURONS, N_UNIQUE_CHARS), device=device)
        self.W2 *= XavierFactor(self.W2)
        self.b2 = torch.zeros((N_UNIQUE_CHARS,), device=device)
        self.params = [self.W1, self.b1, self.W2, self.b2]
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, context):
        # context: [BATCH_SIZE, CONTEXT_WINDOW_SIZE, N_FEATURE_DIMS]
        check_eq(context.shape, [N_BATCHES, CONTEXT_WINDOW_SIZE, N_ATTENTION_HEADS * N_FEATURE_DIMS])
        if DEBUG: print("FORWARD")
        if DEBUG: print(context.shape)
        hidden_output = context @ self.W1 + self.b1
        if DEBUG: print(hidden_output.shape)
        hidden_output = torch.where(hidden_output > 0, hidden_output, 0)
        if DEBUG: print(hidden_output.shape)
        output = hidden_output @ self.W2 + self.b2
        if DEBUG: print(output.shape)
        check_eq(output.shape, [N_BATCHES, CONTEXT_WINDOW_SIZE, N_UNIQUE_CHARS])
        return output

In [29]:
import torch.nn.functional as F

def ResetGrad(params):
    for param in params:
        param.grad = None

def ApplyGrad(params, learning_rate):
    with torch.no_grad():
        for param in params:
            param -= learning_rate * param.grad

class Transformer():
    def __init__(self):
        torch.manual_seed(SEED) # for reproducibility
        self.feature_embedding_table = torch.randn((N_UNIQUE_CHARS, N_FEATURE_DIMS), device=device)
        self.feature_embedding_table *= XavierFactor(self.feature_embedding_table)
        self.feature_mixer = torch.randn((N_ATTENTION_HEADS * N_FEATURE_DIMS, N_ATTENTION_HEADS * N_FEATURE_DIMS), device=device)
        self.feature_mixer *= XavierFactor(self.feature_embedding_table)
        self.attention_heads = [AttentionHead() for _ in range(N_ATTENTION_HEADS)]
        self.feed_forward = FeedForward()
        self.params = [self.feature_embedding_table, self.feature_mixer] + self.feed_forward.params 
        for attention_head in self.attention_heads:
            self.params += attention_head.params
        for param in self.params:
            param.requires_grad = True
        self.positional_encoding = torch.empty((CONTEXT_WINDOW_SIZE, N_FEATURE_DIMS), device=device)
        with torch.no_grad():
            for position in range(CONTEXT_WINDOW_SIZE):
                self.positional_encoding[position] = torch.tensor([
                    math.sin(position / math.pow(10000, 2 * (i // 2) / N_FEATURE_DIMS))
                    if i % 2 == 0
                    else math.cos(position / math.pow(10000, 2 * (i // 2) / N_FEATURE_DIMS))
                    for i in range(N_FEATURE_DIMS)
                ], device=device)
        self.positional_encoding.requires_grad_(False)
        self.learning_rate = LEARNING_RATE
        self.optimizer = torch.optim.Adam(self.params, lr=self.learning_rate)
        self.dropout = nn.Dropout(DROPOUT)
        # self.attended_feature_vectors = torch.empty((N_BATCHES, CONTEXT_WINDOW_SIZE, N_ATTENTION_HEADS * N_FEATURE_DIMS))

    def forward(self, context):
        feature_vectors = self.feature_embedding_table[context]
        feature_vectors = feature_vectors + self.positional_encoding
        attended_feature_vectors = torch.empty((N_BATCHES, CONTEXT_WINDOW_SIZE, N_ATTENTION_HEADS * N_FEATURE_DIMS), device=device)
        for i in range(N_BATCHES):
            attended_feature_vectors[i] = LayerNorm(torch.concat([attention_head.attend(feature_vectors[i]) for attention_head in self.attention_heads], dim=-1));
        if DEBUG: print("Attended feature vectors: ", attended_feature_vectors.shape)
        mixed_feature_vectors = attended_feature_vectors @ self.feature_mixer
        if DEBUG: print("Mixed: ", mixed_feature_vectors.shape)
        mixed_feature_vectors = self.dropout(mixed_feature_vectors)
        if DEBUG: print("Dropout: ", mixed_feature_vectors.shape)
        output = self.feed_forward.forward(LayerNorm(mixed_feature_vectors))
        if DEBUG: print("FF: ", output.shape)
        return output
    
    def backward(self, output, label):
        loss = F.cross_entropy(output, label)
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def summarize(self):
        print("=====ATTENTION=====")
        print(f"For each {N_ATTENTION_HEADS} head, query matrix is {self.attention_heads[0].query_matrix.shape}, key matrix is {self.attention_heads[0].key_matrix.shape}, value matrix is {self.attention_heads[0].value_matrix_up.shape} x {self.attention_heads[0].value_matrix_down.shape}")

In [30]:
transformer = Transformer()
X_batch, Y_batch = GetRandomBatch(train_data)

In [31]:
transformer.summarize()

=====ATTENTION=====
For each 4 head, query matrix is torch.Size([64, 16]), key matrix is torch.Size([64, 16]), value matrix is torch.Size([64, 16]) x torch.Size([16, 64])


In [32]:

for i in range(EPOCHS):
    X_batch, Y_batch = GetRandomBatch(train_data);
    out = transformer.forward(X_batch)
    B, T, C = out.shape
    loss = transformer.backward(out.view(B*T, C), Y_batch.view(B*T))
    if (i % 1 == 0):
        print(f"loss: {loss}")

loss: 4.687386989593506
loss: 4.151747703552246
loss: 3.75696063041687
loss: 3.785055637359619
loss: 3.5693910121917725
loss: 3.493361473083496
loss: 3.4909040927886963
loss: 3.4494805335998535
loss: 3.3883399963378906
loss: 3.576700210571289
loss: 3.336894989013672
loss: 3.456109046936035
loss: 3.4469826221466064
loss: 3.5654749870300293
loss: 3.2703936100006104
loss: 3.5486655235290527
loss: 3.268864154815674
loss: 3.3563315868377686
loss: 3.311411142349243
loss: 3.3018295764923096
loss: 3.2208595275878906
loss: 3.326159954071045
loss: 3.3758790493011475
loss: 3.3416335582733154
loss: 3.447950839996338
loss: 3.1922664642333984
loss: 3.507689952850342
loss: 3.332019805908203
loss: 3.4538211822509766
loss: 3.448734760284424
loss: 3.5187559127807617
loss: 3.2978696823120117
loss: 3.3849141597747803
loss: 3.417086601257324
loss: 3.300497531890869
loss: 3.2631754875183105
loss: 3.2630739212036133
loss: 3.282242774963379
loss: 3.29304575920105
loss: 3.2288074493408203
loss: 3.4414448738098

KeyboardInterrupt: 

In [33]:
def DecodeTokenList(token_list):
    return ''.join(untokenizer_map[int(token)] for token in token_list)


def TestModel(transformer, data, n_tokens=10):
    X_test, _ = GetRandomBatch(data)
    # X_test = X_batch
    context = X_test[0].clone()
    start_context = context.clone()

    print("START:")
    print(DecodeTokenList(context))

    generated_tokens = []

    for i in range(n_tokens):
        # Add fake batch dimension: [T] -> [1, T]
        context_batch = context.view(1, CONTEXT_WINDOW_SIZE)

        # Your transformer currently expects N_BATCHES exactly,
        # so repeat the same context N_BATCHES times.
        context_batch = context_batch.repeat(N_BATCHES, 1)

        # Predict next token from the final position in the first batch row
        with torch.no_grad():
            logits = transformer.forward(context_batch)
        pred = logits[0, -1].argmax(dim=-1)

        generated_tokens.append(pred)

        # Slide context window left and append prediction
        context = torch.cat([context[1:], pred.view(1)], dim=0)

        print(f"PRED {i + 1}: {untokenizer_map[int(pred)]!r}")

    print("\nGENERATED:")
    print(DecodeTokenList(generated_tokens))

    print("\nFULL:")
    print(DecodeTokenList(start_context) + DecodeTokenList(generated_tokens))

TestModel(transformer, train_data, n_tokens=10)

START:
 where he comes: so please you, 
PRED 1: 't'
PRED 2: ' '
PRED 3: 'e'
PRED 4: 'e'
PRED 5: ' '
PRED 6: ' '
PRED 7: ' '
PRED 8: ' '
PRED 9: ' '
PRED 10: 'e'

GENERATED:
t ee     e

FULL:
 where he comes: so please you, t ee     e
